# Understand the micro-watersheds in a tehsil

Join annual groundwater change with terrain composition to see how the selected tehsil is structured before examining one MWS in detail.

Run each cell with **Shift+Enter**. The location controls default to the active KYL tehsil when this notebook is downloaded from CoRE Stack.

In [ ]:
import json, re, sys
from urllib.parse import urlencode
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
import geolibre

GEOSERVER_BASE = "https://geoserver.core-stack.org:8443/geoserver/"
SCOPE = json.loads("{\"state\":\"Jharkhand\",\"district\":\"Dumka\",\"tehsil\":\"Masalia\",\"bounds\":[86.89,23.94,87.24,24.28]}")
LAYER_SPECS = json.loads("[{\"id\":\"mws_layers\",\"label\":\"Micro-watersheds and Hydrological Variables\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"mws_layers\",\"layerNameTemplate\":\"deltaG_well_depth_{district}_{tehsil}\",\"period\":\"2017-2018 to 2024-2025\",\"description\":\"Annual groundwater-storage change and MWS identifiers.\"},{\"id\":\"terrain_vector\",\"label\":\"Terrain Vector\",\"domain\":\"Land\",\"service\":\"WFS\",\"workspace\":\"terrain\",\"layerNameTemplate\":\"{district}_{tehsil}_cluster\",\"period\":\"Current terrain analysis\",\"description\":\"MWS-level plains, slopes, valleys, ridges, hills, and terrain cluster.\"}]")
m = geolibre.connect()
MAP_LAYERS = {}

def geoserver_name(value):
    value = re.sub(r"[()]", "", str(value or "").strip().lower())
    return re.sub(r"_+", "_", re.sub(r"\s+", "_", value)).strip("_")

state_input = widgets.Text(value=SCOPE["state"], description="State:", layout=widgets.Layout(width="98%"))
district_input = widgets.Text(value=SCOPE["district"], description="District:", layout=widgets.Layout(width="98%"))
tehsil_input = widgets.Text(value=SCOPE["tehsil"], description="Tehsil:", layout=widgets.Layout(width="98%"))
display(widgets.VBox([
    widgets.HTML("<b>Study location</b><br><small>Change a name here, then rerun the data cells. No Python editing is needed.</small>"),
    state_input, district_input, tehsil_input,
]))

def selected_scope():
    return {
        "state": state_input.value.strip(),
        "district": geoserver_name(district_input.value),
        "tehsil": geoserver_name(tehsil_input.value),
    }

def get_spec(layer_id):
    return next(layer for layer in LAYER_SPECS if layer["id"] == layer_id)

def layer_url(layer_id, cql_filter=None):
    scope = selected_scope()
    spec = get_spec(layer_id)
    layer_name = spec["layerNameTemplate"].format(**scope)
    qualified = f'{spec["workspace"]}:{layer_name}'
    if spec["service"] == "WFS":
        params = {"service": "WFS", "version": "1.0.0", "request": "GetFeature",
                  "typeName": qualified, "outputFormat": "application/json", "srsName": "EPSG:4326"}
        if cql_filter:
            params["CQL_FILTER"] = cql_filter
        return f'{GEOSERVER_BASE}{spec["workspace"]}/ows?{urlencode(params)}'
    params = {"service": "WCS", "version": "2.0.1", "request": "GetCoverage",
              "CoverageId": qualified, "format": "geotiff", "compression": "LZW"}
    return f'{GEOSERVER_BASE}{spec["workspace"]}/wcs?{urlencode(params)}'

async def fetch_json(url, label="GeoServer layer"):
    try:
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            response = await pyfetch(url)
            if not response.ok:
                raise RuntimeError(f"HTTP {response.status}")
            return await response.json()
        import urllib.request
        with urllib.request.urlopen(url, timeout=90) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as error:
        scope = selected_scope()
        raise RuntimeError(
            f'{label} is not available for {scope["district"]}/{scope["tehsil"]}, or GeoServer could not be reached: {error}'
        ) from error

async def load_geojson(layer_id, cql_filter=None):
    spec = get_spec(layer_id)
    if spec["service"] != "WFS":
        raise ValueError(f'{spec["label"]} is a raster. Use its WCS URL instead of loading it as GeoJSON.')
    data = await fetch_json(layer_url(layer_id, cql_filter), spec["label"])
    if data.get("type") != "FeatureCollection":
        raise RuntimeError(f'{spec["label"]} did not return GeoJSON features.')
    return data

def to_frame(data):
    rows = [dict(feature.get("properties") or {}) for feature in data.get("features", [])]
    return pd.DataFrame(rows)

def uid_column(frame):
    for name in ("uid", "UID", "MWS_UID", "MWS UID"):
        if name in frame.columns:
            return name
    raise KeyError("This layer has no recognised MWS identifier column.")

def with_uid(frame):
    result = frame.copy()
    result["uid"] = result[uid_column(result)].astype(str)
    return result

def numeric(frame, columns):
    return frame.loc[:, columns].apply(pd.to_numeric, errors="coerce")

def json_component(value, component):
    try:
        value = json.loads(value) if isinstance(value, str) else value
        return float(value.get(component)) if isinstance(value, dict) and value.get(component) is not None else np.nan
    except (TypeError, ValueError, json.JSONDecodeError):
        return np.nan

def component_values(frame, columns, component):
    return frame.loc[:, columns].apply(
        lambda series: series.map(lambda value: json_component(value, component))
    )

def features_for_uids(data, uids):
    wanted = {str(uid) for uid in uids}
    names = ("uid", "UID", "MWS_UID", "MWS UID")
    features = []
    for feature in data.get("features", []):
        properties = feature.get("properties") or {}
        value = next((properties.get(name) for name in names if properties.get(name) is not None), None)
        if str(value) in wanted:
            features.append(feature)
    return {"type": "FeatureCollection", "features": features}

def geojson_bounds(data):
    points = []
    def visit(value):
        if isinstance(value, list) and len(value) >= 2 and all(isinstance(v, (int, float)) for v in value[:2]):
            points.append(value[:2])
        elif isinstance(value, list):
            for item in value:
                visit(item)
    for feature in data.get("features", []):
        visit((feature.get("geometry") or {}).get("coordinates", []))
    if not points:
        return None
    xs, ys = zip(*points)
    return [min(xs), min(ys), max(xs), max(ys)]

def show_on_map(key, data, name, **style):
    if not data.get("features"):
        print(f"No features to map for {name}.")
        return None
    previous_layer_id = MAP_LAYERS.get(key)
    if previous_layer_id:
        try:
            m.remove_layer(previous_layer_id)
        except Exception:
            pass
    MAP_LAYERS[key] = m.add_geojson(data, name=name, **style)
    bounds = geojson_bounds(data)
    if bounds:
        m.fit_bounds(bounds)
    return MAP_LAYERS[key]

def year_columns(frame, prefix="", pattern=r"^\d{4}_\d{4}$"):
    return sorted(column for column in frame.columns if column.startswith(prefix) and re.search(pattern, column))

print(f'Ready for {SCOPE["tehsil"]}, {SCOPE["district"]}.')

In [ ]:
def build_overview(mws_frame, terrain_frame):
    mws = with_uid(mws_frame).set_index("uid")
    terrain = with_uid(terrain_frame).set_index("uid")
    annual = [column for column in mws.columns if re.match(r"^\d{4}_\d{4}$", column)]
    result = pd.DataFrame(index=mws.index)
    result["Area (ha)"] = pd.to_numeric(mws.get("area_in_ha"), errors="coerce")
    result["Mean annual groundwater change"] = component_values(mws, annual, "DeltaG").mean(axis=1)
    result["Recent net groundwater change"] = pd.to_numeric(mws.get("Net2020_25"), errors="coerce")
    for source, label in [("plain_area", "Plains"), ("slopy_area", "Slopes"),
                          ("valley_are", "Valleys"), ("ridge_area", "Ridges"),
                          ("hill_slope", "Hills")]:
        result[label + " (%)"] = pd.to_numeric(terrain.get(source), errors="coerce").reindex(result.index)
    result.index.name = "MWS UID"
    return result

def overview_summary(profile):
    return pd.DataFrame({
        "Measure": ["Micro-watersheds", "Mapped area (ha)", "Median annual groundwater change", "MWS with negative recent net change"],
        "Value": [len(profile), round(profile["Area (ha)"].sum(), 1),
                  round(profile["Mean annual groundwater change"].median(), 2),
                  int((profile["Recent net groundwater change"] < 0).sum())],
    })

def plot_overview(profile):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    values = profile["Mean annual groundwater change"].dropna()
    axes[0].hist(values, bins=min(12, max(5, len(values) // 5)), color="#2563eb", edgecolor="white")
    axes[0].axvline(0, color="#991b1b", linewidth=1.5, label="No net change")
    axes[0].set(title="Groundwater-change distribution", xlabel="Mean annual change", ylabel="Number of MWS")
    axes[0].legend()
    terrain_columns = ["Plains (%)", "Slopes (%)", "Valleys (%)", "Ridges (%)", "Hills (%)"]
    weighted = profile[terrain_columns].mul(profile["Area (ha)"], axis=0).sum().div(profile["Area (ha)"].sum())
    weighted.sort_values().plot.barh(ax=axes[1], color="#65a30d")
    axes[1].set(title="Area-weighted terrain composition", xlabel="Share of mapped area (%)", ylabel="")
    plt.tight_layout()
    plt.show()

## 1. Load only the two relevant layers

The join uses the published MWS UID. Missing rows remain visible rather than being silently filled.

In [ ]:
mws_geojson = await load_geojson("mws_layers")
terrain_geojson = await load_geojson("terrain_vector")
mws = to_frame(mws_geojson)
terrain = to_frame(terrain_geojson)
profile = build_overview(mws, terrain)
display(overview_summary(profile))
display(profile.round(2).head(10))

## 2. Read the tehsil as a whole

The histogram preserves variation between MWSes; the terrain chart uses mapped area rather than treating differently sized watersheds as equal.

In [ ]:
plot_overview(profile)

## 3. Return the evidence to the map

All MWS polygons are added as a temporary notebook layer. Use GeoLibre identify or its attribute table to inspect one.

In [ ]:
show_on_map(
    "overview-mws", mws_geojson, "Notebook · MWS overview",
    fillColor="#60a5fa", strokeColor="#1e3a8a", fillOpacity=0.28,
)

## Interpretation

A tehsil average is context, not a description of every MWS. Negative groundwater change identifies a measured direction in the published series; it does not by itself establish the cause.

## Optional: check completeness

In [ ]:
completeness = profile.notna().mean().mul(100).round(1).sort_values()
display(completeness.rename("Populated values (%)").to_frame())